In [1]:
# base
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic() 
model = "claude-haiku-4-5"

In [2]:
# Helpers
def add_user_message(messages, text):
  user_message = {
    "role": "user", 
    "content": text
  }
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {
    "role": "assistant",
    "content": text
  }
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
  params= {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)
  return message.content[0].text

In [3]:
# generate dataset
import json

def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [4]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
  messages = []
  add_user_message(messages, prompt)
  # TODO: need to see if it'd be necessary to add these pre-filling + stop sequences since it doesn't exists in the previous tutorial
  # add_assistant_message(messages, '```code')
  output = chat(
    messages,
    # stop_sequences=["```"]
  )
  return output

In [5]:
# grade_by_model from text based guide, high possibility of returning error/wrong since it's using like completely different variables
# replaced previously {task} and {solution} to {test_case} and {output} accordingly. Might need to test this function too
def grade_by_model_0(test_case, output):
  eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strenghts": An array of 1-3 key strength
- "weaknesses": An array of 1-3 key areas of improvements
- "reasoning" "A concise explanation of your assessment
- "score": A number between 1-10
"""

  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")

  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

In [6]:
def grade_by_model(test_case, output):
  eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strenghts
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
  "strengths": string[],
  "weaknesses": string[],
  "reasoning": string,
  "score": number
}}
"""
  
  messages = []
  add_user_message(messages, eval_prompt)
  add_assistant_message(messages, "```json")
  eval_text = chat(messages, stop_sequences=["```"])
  return json.loads(eval_text)

In [15]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the result"""
  output = run_prompt(test_case)
  
  model_grade = grade_by_model(test_case, output)

  return {
    "output": output,
    "test_case": test_case,
    "score": model_grade["score"],
    "reasoning": model_grade["reasoning"],
    # can skip strength & weakness to avoid being too long
    # "strength": model_grade["strengths"],
    # "weaknesses": model_grade["weaknesses"]
  }



In [18]:
from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result["score"] for result in results])
  print(f"Average score: {average_score}")


  return results

In [19]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [14]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses the bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/key/path'\n    \n    Returns:\n        The bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \n    Examples:\n        >>> parse_s3_bucket_name('s3://my-bucket/path/to/key')\n        'my-bucket'\n        >>> parse_s3_bucket_name('s3://bucket-with-dashes/folder/file.txt')\n        'bucket-with-dashes'\n    \"\"\"\n    if not isinstance(s3_uri, str):\n        raise ValueError(\"Input must be a string\")\n    \n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"URI must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' and get the first part (bucket name)\n    parts 